In [ ]:
import decoupler as dc
import pandas as pd
import omnipath as op
import numpy as np
import mygene
import torch
import anndata as ad

## RNA GRN Prior: DoRothEA

In [ ]:
dorothea = pd.read_csv("dorothea_AB_priors.csv")
dorothea

In [ ]:
rna_raw = pd.read_csv("rnaseq_raw_filtered.csv", index_col = 0)
rna_raw

In [ ]:
mg = mygene.MyGeneInfo()
unique_symbols = list(set(dorothea['source']).union(set(dorothea['target'])))
mapping = mg.querymany(unique_symbols, scopes='symbol', fields='ensembl.gene', species='human', verbose=False)

symbol_to_ens = {}
for entry in mapping:
    if 'ensembl' in entry:
        if isinstance(entry['ensembl'], list):
            symbol_to_ens[entry['query']] = entry['ensembl'][0]['gene']
        elif isinstance(entry['ensembl'], dict):
            symbol_to_ens[entry['query']] = entry['ensembl']['gene']

dorothea['source_ens'] = dorothea['source'].map(symbol_to_ens)
dorothea['target_ens'] = dorothea['target'].map(symbol_to_ens)

valid_genes = set(rna_raw.columns)
dorothea_valid = dorothea[
    dorothea['source_ens'].isin(valid_genes) & 
    dorothea['target_ens'].isin(valid_genes)
].drop_duplicates(subset=['source_ens', 'target_ens'])

unique_tfs = sorted(dorothea_valid['source_ens'].unique())
tf_indices = [rna_raw.columns.get_loc(tf) for tf in unique_tfs]

prior_mask = torch.zeros((len(tf_indices), len(rna_raw.columns)))

tf_to_row = {tf: i for i, tf in enumerate(unique_tfs)}
gene_to_col = {gene: i for i, gene in enumerate(rna_raw.columns)}

for _, row in dorothea_valid.iterrows():
    r = tf_to_row[row['source_ens']]
    c = gene_to_col[row['target_ens']]
    prior_mask[r, c] = 1.0

In [ ]:
save_dict = {
    'prior_mask': prior_mask,  
    'tf_indices': tf_indices   
}

torch.save(save_dict, 'rna_priors.pt')

In [ ]:
rna_priors_dorothea = torch.load("rna_priors.pt")
rna_priors_dorothea["prior_mask"].shape

## RNA Prior V2: OmniPath CollecTRI

In [ ]:
rna_citeseq = ad.read_h5ad("data/pbmc_citeseq/rna_final_processed.h5ad")
rna_citeseq

In [ ]:
rna_citeseq = ad.read_h5ad("data/combined/rna_combined.h5ad")
rna_citeseq

In [ ]:
rna_citeseq.var_names

In [ ]:
collectri = dc.op.collectri(organism='human')
print(f"Downloaded {len(collectri)} interactions.")
print(collectri.head())

In [ ]:
target_genes = rna_citeseq.var_names.tolist()
gene_to_idx = {gene: i for i, gene in enumerate(target_genes)}
n_genes = len(target_genes)

prior_mask = torch.zeros((n_genes, n_genes))

valid_edges = collectri[collectri['source'].isin(target_genes) & collectri['target'].isin(target_genes)]

if not valid_edges.empty:
    source_indices = [gene_to_idx[g] for g in valid_edges['source']]
    target_indices = [gene_to_idx[g] for g in valid_edges['target']]
    prior_mask[source_indices, target_indices] = 1.0

torch.save({
    'prior_mask': prior_mask,
    'hvg_columns': target_genes
}, 'rna_hvg_priors_collectri_sc_combined.pt')

In [ ]:
prior_mask.shape

In [ ]:
sum(prior_mask.flatten() == 1) / len(prior_mask.flatten())

In [ ]:
library_sizes = rna_raw.sum(axis=1)
r_cpm = rna_raw.div(library_sizes, axis=0) * 1e6    
r_log = np.log2(r_cpm + 1)
r_vars = r_log.var(axis=0)
hvg_indices = np.argsort(r_vars)[-5000:]
hvg_names = rna_raw.columns[hvg_indices] # The 5000 gene names in order

# 2. Map CollecTRI Symbols to Ensembl
mg = mygene.MyGeneInfo()
unique_symbols = list(set(collectri['source']).union(set(collectri['target'])))
mapping = mg.querymany(unique_symbols, scopes='symbol', fields='ensembl.gene', species='human', verbose=False)

symbol_to_ens = {}
for entry in mapping:
    if 'ensembl' in entry:
        if isinstance(entry['ensembl'], list):
            symbol_to_ens[entry['query']] = entry['ensembl'][0]['gene']
        elif isinstance(entry['ensembl'], dict):
            symbol_to_ens[entry['query']] = entry['ensembl']['gene']

collectri['source_ens'] = collectri['source'].map(symbol_to_ens)
collectri['target_ens'] = collectri['target'].map(symbol_to_ens)

# 3. Filter CollecTRI for genes present in the HVGs
valid_hvg_set = set(hvg_names)
collectri_filtered = collectri[
    collectri['source_ens'].isin(valid_hvg_set) & 
    collectri['target_ens'].isin(valid_hvg_set)
].drop_duplicates(subset=['source_ens', 'target_ens'])

# 4. Build the Square Prior Mask (5000 x 5000)
num_genes = len(hvg_names)
prior_mask_rna = torch.zeros((num_genes, num_genes))

# Map every gene to its index (0 to 4999)
gene_to_idx = {gene: i for i, gene in enumerate(hvg_names)}

for _, row in collectri_filtered.iterrows():
    r = gene_to_idx[row['source_ens']]
    c = gene_to_idx[row['target_ens']]
    prior_mask_rna[r, c] = 1.0

# 5. Save
save_dict = {
    'prior_mask': prior_mask_rna,
    'hvg_columns': hvg_names.tolist() 
}

torch.save(save_dict, 'rna_hvg_priors_collectri.pt')
print(f"Saved Square Prior Mask with shape: {prior_mask_rna.shape}")

In [ ]:
save_dict["prior_mask"].shape

In [ ]:
save_dict

## Proteomics Prior: OmniPath

In [ ]:
interactions = op.interactions.OmniPath.get(
    organism='human',
    directed=True,
    genesymbols=True
)

# 2. Select relevant columns
# is_stimulation and is_inhibition can be useful if your graph handles sign
prot_priors = interactions[['source', 'target', 'is_stimulation', 'is_inhibition']]

In [ ]:
protein_citeseq = ad.read_h5ad("data/pbmc_citeseq/cite_seq_processed.h5ad")
protein_citeseq

In [ ]:
protein_citeseq = ad.read_h5ad("data/combined/citeseq_combined.h5ad")
protein_citeseq

In [ ]:
protein_citeseq.var_names

In [ ]:
interactions = op.interactions.OmniPath.get(organism='human', genesymbols=True)
interactions

In [ ]:
interactions.columns

In [ ]:
target_proteins = protein_citeseq.var_names.tolist()

mg = mygene.MyGeneInfo()
query_results = mg.querymany(target_proteins, scopes='symbol,alias,name', fields='symbol', species='human', verbose=False)

name_map = {}
for res in query_results:
    if 'symbol' in res:
        name_map[res['query']] = res['symbol']

# 3. Build Mask using GENESYMBOL columns
n_proteins = len(target_proteins)
prior_mask = torch.zeros((n_proteins, n_proteins))

# We use source_genesymbol instead of source
valid_interactions = interactions[
    interactions['source_genesymbol'].isin(name_map.values()) & 
    interactions['target_genesymbol'].isin(name_map.values())
]

prot_to_idx = {prot: i for i, prot in enumerate(target_proteins)}

for _, row in valid_interactions.iterrows():
    # USE THE GENESYMBOL COLUMNS
    src_symbol = row['source_genesymbol']
    tgt_symbol = row['target_genesymbol']
    
    # Reverse lookup: Find which CITE-seq ID maps to this symbol
    src_original_names = [k for k, v in name_map.items() if v == src_symbol]
    tgt_original_names = [k for k, v in name_map.items() if v == tgt_symbol]
    
    for src in src_original_names:
        for tgt in tgt_original_names:
            if src in prot_to_idx and tgt in prot_to_idx:
                prior_mask[prot_to_idx[src], prot_to_idx[tgt]] = 1.0

torch.save({
    'prior_mask': prior_mask,
    'protein_names': target_proteins,
    'mapping': name_map
}, 'protein_priors_sc_combined.pt')

In [ ]:
prior_mask.shape

In [ ]:
sum(prior_mask.flatten() == 1) / len(prior_mask.flatten())

In [ ]:
proteomics = pd.read_csv("proteomics_raw_filtered.csv", index_col = 0)
proteomics

In [ ]:
proteomics_metadata = pd.read_csv("proteomics_metadata_with_uniprot.csv")
proteomics_metadata

In [ ]:
prot_priors_filtered = prot_priors[(prot_priors["source"].isin(list(proteomics_metadata["uniprot_id"]))) & (prot_priors["target"].isin(list(proteomics_metadata["uniprot_id"])))]
prot_priors_filtered

In [ ]:
prot_features = proteomics.columns.tolist()
feature_to_idx = {name: i for i, name in enumerate(prot_features)}
num_features = len(prot_features)

uniprot_to_indices = {}
valid_meta = proteomics_metadata.dropna(subset=['uniprot_id'])

for _, row in valid_meta.iterrows():
    p_name = row['protein_name']
    u_id = row['uniprot_id']
    if p_name in feature_to_idx:
        idx = feature_to_idx[p_name]
        if u_id not in uniprot_to_indices:
            uniprot_to_indices[u_id] = []
        uniprot_to_indices[u_id].append(idx)

prior_mask_p = torch.zeros((num_features, num_features))

for _, row in prot_priors_filtered.iterrows():
    src_uid = row['source']
    tgt_uid = row['target']
    
    if src_uid in uniprot_to_indices and tgt_uid in uniprot_to_indices:
        src_indices = uniprot_to_indices[src_uid]
        tgt_indices = uniprot_to_indices[tgt_uid]
        
        for s_idx in src_indices:
            for t_idx in tgt_indices:
                prior_mask_p[s_idx, t_idx] = 1.0

save_dict_p = {
    'prior_mask': prior_mask_p,
    'tf_indices': None 
}

torch.save(save_dict_p, 'protein_priors.pt')

In [ ]:
prior_mask_p

## Metabolomics Prior: REACTOME

In [ ]:
metabolomics_metadata = pd.read_csv("metabolomics_metadata_with_smiles.csv")
metabolomics_metadata

In [ ]:
chebi_reactions = pd.read_csv("ChEBI2ReactomeReactions.tsv", sep = "\t", header = None)
chebi_reactions

In [ ]:
metabolite_name_map = pd.read_csv("metabolite_name_map.csv")
metabolite_name_map

In [ ]:
chebi_reactions.columns = ['chebi_id', 'reaction_id', 'url', 'rxn_name', 'evidence', 'species']
reactome_human = chebi_reactions[chebi_reactions['species'] == 'Homo sapiens'].copy()

reactome_human['chebi_clean'] = reactome_human['chebi_id'].astype(str).str.replace('CHEBI:', '').str.split('.').str[0]

valid_map = metabolite_name_map.dropna(subset=['HMDB', 'ChEBI']).copy()
valid_map['chebi_clean'] = valid_map['ChEBI'].astype(str).str.split('.').str[0]
hmdb_to_chebi = dict(zip(valid_map['HMDB'], valid_map['chebi_clean']))


chebi_to_indices = {}

for idx, row in metabolomics_metadata.iterrows():
    hmdb = row['hmdb_id']
    if pd.notna(hmdb) and hmdb in hmdb_to_chebi:
        chebi = hmdb_to_chebi[hmdb]
        
        if chebi not in chebi_to_indices:
            chebi_to_indices[chebi] = []
        chebi_to_indices[chebi].append(idx)

num_features = len(metabolomics_metadata)
prior_mask_m = torch.zeros((num_features, num_features))


rxn_groups = reactome_human.groupby('reaction_id')['chebi_clean'].apply(list)

for rxn_chebis in rxn_groups:
    involved_indices = []
    for chebi in rxn_chebis:
        if chebi in chebi_to_indices:
            involved_indices.extend(chebi_to_indices[chebi])
    
    
    involved_indices = list(set(involved_indices)) 
    for i in involved_indices:
        for j in involved_indices:
            if i != j: # No self-loops
                prior_mask_m[i, j] = 1.0

save_dict_m = {
    'prior_mask': prior_mask_m,
    'tf_indices': None 
}

torch.save(save_dict_m, 'metabolomics_priors.pt')

In [ ]:
prior_mask_m.shape